In [1]:
!nvidia-smi

Sun Aug 16 13:24:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd


In [3]:
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 67 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (310 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122492 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [4]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [5]:
!apt-get install -y pciutils lshw

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids usb.ids
The following NEW packages will be installed:
  libpci3 lshw pci.ids pciutils usb.ids
0 upgraded, 5 newly installed, 0 to remove and 67 not upgraded.
Need to get 884 kB of archives.
After this operation, 3,256 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 lshw amd64 02.19.git.2021.06.19.996aaad9c7-2ubuntu0.22.04.1 [322 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/main amd64 usb.ids all 2022.04.02-1 [219 kB]
Fetched 884 kB in 2s (384 kB/s)
Selecting previously unse

In [6]:
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama --version

ollama version is 0.32.13


In [7]:
!ollama pull qwen2.5:7b

In [8]:
!ollama run qwen2.5:7b "Say OK" --verbose
!ollama ps

OK! Is there anything specific you would like to discuss or ask about?

total duration:       1m28.631722684s
load duration:        41.449060508s
prompt eval count:    31 token(s)
prompt eval duration: 37.021886s
prompt eval rate:     0.84 tokens/s
eval count:           16 token(s)
eval duration:        10.154616s
eval rate:            1.58 tokens/s
NAME          ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
qwen2.5:7b    845dbda0ea48    4.7 GB    100% GPU     4096       4 minutes from now    


In [9]:
!ollama run qwen2.5:7b "Name five colours, one per line." --verbose

Red
Blue
Green
Yellow
Purple

total duration:       732.17389ms
load duration:        374.663341ms
prompt eval count:    37 token(s)
prompt eval duration: 64.685ms
prompt eval rate:     572.00 tokens/s
eval count:           10 token(s)
eval duration:        289.842ms
eval rate:            34.50 tokens/s


In [11]:
from google.colab import files
televerse = files.upload()      # sélectionne data/processed/dry_run_sample.json

Saving dry_run_sample.json to dry_run_sample.json


In [4]:
PROMPT = """You are a biomedical relation extraction system.

Extract relations from the abstract below.

ENTITY TYPES — use exactly these labels:
- Drug: a therapeutic agent administered to a patient
- Protein: a protein, receptor, enzyme or transporter
- ChemicalSubstance: any other molecule (endogenous compound, metabolite, hormone)
- BiologicalProcess: a biological process, pathway or molecular activity
- PhenotypicFeature: an observable symptom or clinical sign
- Disease: a named pathology or diagnosis
- OrganismTaxon: an organism (bacterium, virus, parasite)

RELATION TYPES — you MUST use one of these exact labels. Never use a verb
taken from the text as a label.

VERB TRANSLATION TABLE — find the verb in the text, then output the label:
  inhibits, blocks, antagonizes, suppresses, is an antagonist of,
  reduces the activity of              -> decreases activity of
  activates, stimulates, agonizes, is an agonist of,
  enhances the activity of             -> increases activity of
  increases the level / production / release of, upregulates the amount of
                                       -> increases abundance of
  decreases the level / production / release of, depletes
                                       -> decreases abundance of
  promotes, drives, upregulates (a process)
                                       -> positively regulates
  downregulates, impairs, dampens (a process)
                                       -> negatively regulates
  is associated with higher levels or greater severity
                                       -> positively correlated with
  is associated with lower levels, protects against
                                       -> negatively correlated with
  is involved in, takes part in, is a component of
                                       -> participates in
  is a symptom of, is a sign of, manifests as
                                       -> manifestation of
  leads to, results in, is responsible for
                                       -> causes

Use "decreases activity of" / "increases activity of" when the target is a
Protein. Use "negatively regulates" / "positively regulates" when the target
is a BiologicalProcess.

RULES:
1. Extract ONLY relations explicitly stated in the abstract. Never infer.
2. Write entity names exactly as they appear in the text. Do not expand abbreviations.
3. Preserve direction: the source performs the relation on the target.
4. Never invert polarity. An antagonist decreases; an agonist increases.
5. If several drugs are discussed, attribute each relation to the drug actually
   named in that statement.
6. If no relation matches the allowed labels, return an empty list.

Return JSON only, in this exact shape:
{{"relations": [{{"source": "", "source_type": "", "relation": "",
                  "target": "", "target_type": ""}}]}}

ABSTRACT:
{texte}
"""

In [12]:
!ls -la /content/*.json

-rw-r--r-- 1 root root 18462 Aug 16 17:22 /content/dry_run_sample.json


In [5]:
import json, time, requests

with open("dry_run_sample.json", encoding="utf-8") as fichier:
    echantillon = json.load(fichier)

def extraire(texte):
    debut = time.time()
    reponse = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json={
            "model": "qwen2.5:7b",
            "prompt": PROMPT.format(texte=texte),
            "format": "json",
            "stream": False,
            "options": {"temperature": 0, "num_ctx": 4096},
        },
        timeout=600,
    )
    return reponse.json()["response"], time.time() - debut

resultats, durees = [], []
for i, article in enumerate(echantillon, 1):
    texte = article["titre"] + "\n\n" + article["resume"]
    brut, duree = extraire(texte)
    durees.append(duree)
    try:
        relations = json.loads(brut).get("relations", [])
        erreur = None
    except Exception as e:
        relations, erreur = [], str(e)
    resultats.append({"pmid": article["pmid"], "annee": article["annee"],
                      "titre": article["titre"], "duree": duree,
                      "relations": relations, "erreur": erreur,
                      "brut": brut,
                      "attendues": [p for p in article["provenance"]
                                    if p["role"] == "cible"]})
    print(f"{i}/{len(echantillon)}  PMID {article['pmid']}  "
          f"{duree:.1f}s  {len(relations)} relation(s)"
          f"{'  ERREUR' if erreur else ''}")

print(f"\nDurée moyenne : {sum(durees)/len(durees):.1f} s/abstract")
print(f"Projection sur 483 abstracts : "
      f"{sum(durees)/len(durees)*483/60:.0f} minutes")

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7f40720b7b30>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [14]:
import subprocess, time, requests

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    start_new_session=True,          # détache du groupe de processus du noyau
)

for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        print("Serveur prêt")
        break
    except Exception:
        time.sleep(1)
else:
    print("Le serveur n'a pas démarré")

Serveur prêt


In [15]:
!ollama list

NAME          ID              SIZE      MODIFIED       
qwen2.5:7b    845dbda0ea48    4.7 GB    51 minutes ago    


In [16]:
try:
    requests.get("http://127.0.0.1:11434/api/tags", timeout=5)
except Exception:
    raise RuntimeError("Ollama ne répond pas — relancer la cellule du serveur")
import json, time, requests

with open("dry_run_sample.json", encoding="utf-8") as fichier:
    echantillon = json.load(fichier)

def extraire(texte):
    debut = time.time()
    reponse = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json={
            "model": "qwen2.5:7b",
            "prompt": PROMPT.format(texte=texte),
            "format": "json",
            "stream": False,
            "options": {"temperature": 0, "num_ctx": 4096},
        },
        timeout=600,
    )
    return reponse.json()["response"], time.time() - debut

resultats, durees = [], []
for i, article in enumerate(echantillon, 1):
    texte = article["titre"] + "\n\n" + article["resume"]
    brut, duree = extraire(texte)
    durees.append(duree)
    try:
        relations = json.loads(brut).get("relations", [])
        erreur = None
    except Exception as e:
        relations, erreur = [], str(e)
    resultats.append({"pmid": article["pmid"], "annee": article["annee"],
                      "titre": article["titre"], "duree": duree,
                      "relations": relations, "erreur": erreur,
                      "brut": brut,
                      "attendues": [p for p in article["provenance"]
                                    if p["role"] == "cible"]})
    print(f"{i}/{len(echantillon)}  PMID {article['pmid']}  "
          f"{duree:.1f}s  {len(relations)} relation(s)"
          f"{'  ERREUR' if erreur else ''}")

print(f"\nDurée moyenne : {sum(durees)/len(durees):.1f} s/abstract")
print(f"Projection sur 483 abstracts : "
      f"{sum(durees)/len(durees)*483/60:.0f} minutes")

1/8  PMID 16597316  4.8s  0 relation(s)
2/8  PMID 19108785  1.3s  0 relation(s)
3/8  PMID 19169095  2.9s  2 relation(s)
4/8  PMID 34540303  1.9s  1 relation(s)
5/8  PMID 27217448  1.3s  0 relation(s)
6/8  PMID 25667920  8.7s  5 relation(s)
7/8  PMID 21443098  6.8s  5 relation(s)
8/8  PMID 6225313  2.0s  1 relation(s)

Durée moyenne : 3.7 s/abstract
Projection sur 483 abstracts : 30 minutes


In [16]:
TYPES = {"Drug", "Protein", "ChemicalSubstance", "BiologicalProcess",
         "PhenotypicFeature", "Disease", "OrganismTaxon"}
RELATIONS = {"positively regulates", "negatively regulates",
             "increases activity of", "decreases activity of",
             "increases abundance of", "positively correlated with",
             "negatively correlated with", "participates in",
             "manifestation of", "causes"}

hors_schema = []
for r in resultats:
    print(f"\n{'=' * 74}")
    print(f"PMID {r['pmid']} ({r['annee']}) — {r['duree']:.1f}s")
    print(f"{r['titre'][:72]}")

    print("\n  ATTENDU :")
    for a in r["attendues"]:
        print(f"    {a['source'][:26]} --{a['relation']}--> {a['cible'][:26]}")

    print("\n  EXTRAIT :")
    if r["erreur"]:
        print(f"    ERREUR DE PARSING : {r['erreur']}")
        print(f"    brut : {r['brut'][:200]}")
    for rel in r["relations"]:
        st, tt = rel.get("source_type"), rel.get("target_type")
        rl = rel.get("relation")
        drapeaux = []
        if st not in TYPES: drapeaux.append(f"type source '{st}'")
        if tt not in TYPES: drapeaux.append(f"type cible '{tt}'")
        if rl not in RELATIONS: drapeaux.append(f"relation '{rl}'")
        marque = "  ⚠ " + ", ".join(drapeaux) if drapeaux else ""
        if drapeaux:
            hors_schema.append(rel)
        print(f"    [{st}] {str(rel.get('source'))[:24]} "
              f"--{rl}--> [{tt}] {str(rel.get('target'))[:24]}{marque}")

total = sum(len(r["relations"]) for r in resultats)
echecs = sum(1 for r in resultats if r["erreur"])
print(f"\n{'=' * 74}")
print(f"Relations extraites : {total}")
print(f"Hors schéma          : {len(hors_schema)}")
print(f"Échecs de parsing    : {echecs}/{len(resultats)}")


PMID 8521760 (1995) — 6.1s
Alteplase. A reappraisal of its pharmacology and therapeutic use in vasc

  ATTENDU :
    fibrinolysis --negatively correlated with--> Pulmonary thromboembolism

  EXTRAIT :
    [Drug] alteplase --stimulates--> [BiologicalProcess] fibrinolysis of blood th  ⚠ relation 'stimulates'

PMID 19169095 (2008) — 2.8s
Finasteride-induced myalgia and HyperCKemia.

  ATTENDU :
    Finasteride --decreases activity of--> 3-oxo-5-alpha-steroid 4-de
    3-oxo-5-alpha-steroid 4-de --increases abundance of--> Dihydrotestosterone

  EXTRAIT :
    [Drug] Finasteride --causes--> [PhenotypicFeature] myalgia
    [Drug] Finasteride --causes--> [PhenotypicFeature] hyperCKemia

PMID 35947166 (2022) — 6.8s
Psychopharmacological properties and therapeutic profile of the antidepr

  ATTENDU :
    venlafaxine --negatively regulates--> Sodium-dependent noradrena
    Sodium-dependent noradrena --positively regulates--> norepinephrine uptake
    venlafaxine --negatively regulates--> Sodium-

In [2]:
import requests, os
print("Fichiers :", [f for f in os.listdir("/content") if f.endswith(".json")])
try:
    r = requests.get("http://127.0.0.1:11434/api/tags", timeout=3)
    print("Ollama : actif —", [m["name"] for m in r.json().get("models", [])])
except Exception:
    print("Ollama : ARRÊTÉ → relancer la cellule du serveur")

Fichiers : []
Ollama : ARRÊTÉ → relancer la cellule du serveur


In [3]:
import subprocess, time, requests

# 1. Prérequis système
!apt-get install -y -qq zstd pciutils lshw

# 2. Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Serveur, détaché du noyau
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    start_new_session=True,
)
for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        print("Serveur prêt")
        break
    except Exception:
        time.sleep(1)
else:
    print("Le serveur n'a pas démarré")

# 4. Modèle (instantané s'il est déjà présent)
!ollama pull qwen2.5:7b
!ollama list

Selecting previously unselected package pci.ids.
(Reading database ... 122492 files and directories currently installed.)
Preparing to unpack .../0-pci.ids_0.0~2022.01.22-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2022.01.22-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../1-libpci3_1%3a3.7.0-6_amd64.deb ...
Unpacking libpci3:amd64 (1:3.7.0-6) ...
Selecting previously unselected package lshw.
Preparing to unpack .../2-lshw_02.19.git.2021.06.19.996aaad9c7-2ubuntu0.22.04.1_amd64.deb ...
Unpacking lshw (02.19.git.2021.06.19.996aaad9c7-2ubuntu0.22.04.1) ...
Selecting previously unselected package pciutils.
Preparing to unpack .../3-pciutils_1%3a3.7.0-6_amd64.deb ...
Unpacking pciutils (1:3.7.0-6) ...
Selecting previously unselected package usb.ids.
Preparing to unpack .../4-usb.ids_2022.04.02-1_all.deb ...
Unpacking usb.ids (2022.04.02-1) ...
Selecting previously unselected package zstd.
Preparing to unpack .../5-zstd_1.4.8+dfsg-3build1_amd6

In [4]:
!nvidia-smi

Sun Aug 16 16:41:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!ls -la /content/*.json

-rw-r--r-- 1 root root 22076 Aug 16 16:45 /content/dry_run_sample.json


In [1]:
with open("controle_imatinib.json", encoding="utf-8") as fichier:
    controle = json.load(fichier)

print(f"Articles à contrôler : {len(controle)}\n")
for article in controle:
    texte = article["titre"] + "\n\n" + article["resume"]
    brut, duree = extraire(texte)
    try:
        relations = json.loads(brut).get("relations", [])
    except Exception:
        relations = []
    attendues = {(p["source"], p["relation"], p["cible"])
                 for p in article["provenance"] if p["role"] == "cible"}
    print(f"{'=' * 70}")
    print(f"PMID {article['pmid']} ({article['annee']}) — {duree:.1f}s")
    print(f"{article['titre'][:66]}")
    print("  ATTENDU :")
    for s, r, c in attendues:
        print(f"    {s[:24]} --{r}--> {c[:24]}")
    print("  EXTRAIT :")
    for rel in relations:
        print(f"    [{rel.get('source_type')}] {str(rel.get('source'))[:22]} "
              f"--{rel.get('relation')}--> "
              f"[{rel.get('target_type')}] {str(rel.get('target'))[:22]}")
    if not relations:
        print("    (aucune)")

NameError: name 'json' is not defined

In [6]:
import subprocess, time, requests, os

def ollama_repond():
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not os.path.exists("/usr/local/bin/ollama"):
    print("Installation des prérequis et d'Ollama (~2 min)...")
    subprocess.run("apt-get install -y -qq zstd pciutils lshw",
                   shell=True, check=False)
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                   shell=True, check=False)

if not ollama_repond():
    print("Démarrage du serveur...")
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                     start_new_session=True)
    for _ in range(30):
        if ollama_repond():
            break
        time.sleep(1)

print("Ollama :", "actif" if ollama_repond() else "ARRÊTÉ")
subprocess.run("ollama pull qwen2.5:7b", shell=True, check=False)
subprocess.run("ollama list", shell=True, check=False)

Installation des prérequis et d'Ollama (~2 min)...
Démarrage du serveur...
Ollama : actif


CompletedProcess(args='ollama list', returncode=0)

In [7]:
import requests
r = requests.get("http://127.0.0.1:11434/api/tags", timeout=5)
modeles = [m["name"] for m in r.json().get("models", [])]
print("Modèles disponibles :", modeles)

Modèles disponibles : ['qwen2.5:7b']


In [8]:
PROMPT = """You are a biomedical relation extraction system.

Extract relations from the abstract below.

ENTITY TYPES — use exactly these labels:
- Drug: a therapeutic agent administered to a patient
- Protein: a protein, receptor, enzyme or transporter
- ChemicalSubstance: any other molecule (endogenous compound, metabolite, hormone)
- BiologicalProcess: a biological process, pathway or molecular activity
- PhenotypicFeature: an observable symptom or clinical sign
- Disease: a named pathology or diagnosis
- OrganismTaxon: an organism (bacterium, virus, parasite)

RELATION TYPES — you MUST use one of these exact labels. Never use a verb
taken from the text as a label.

VERB TRANSLATION TABLE — find the verb in the text, then output the label:
  inhibits, blocks, antagonizes, suppresses, is an antagonist of,
  reduces the activity of              -> decreases activity of
  activates, stimulates, agonizes, is an agonist of,
  enhances the activity of             -> increases activity of
  increases the level / production / release of, upregulates the amount of
                                       -> increases abundance of
  decreases the level / production / release of, depletes
                                       -> decreases abundance of
  promotes, drives, upregulates (a process)
                                       -> positively regulates
  downregulates, impairs, dampens (a process)
                                       -> negatively regulates
  is associated with higher levels or greater severity
                                       -> positively correlated with
  is associated with lower levels, protects against
                                       -> negatively correlated with
  is involved in, takes part in, is a component of
                                       -> participates in
  is a symptom of, is a sign of, manifests as
                                       -> manifestation of
  leads to, results in, is responsible for
                                       -> causes

Use "decreases activity of" / "increases activity of" when the target is a
Protein. Use "negatively regulates" / "positively regulates" when the target
is a BiologicalProcess.

RULES:
1. Extract ONLY relations explicitly stated in the abstract. Never infer.
2. Write entity names exactly as they appear in the text. Do not expand abbreviations.
3. Preserve direction: the source performs the relation on the target.
4. Never invert polarity. An antagonist decreases; an agonist increases.
5. If several drugs are discussed, attribute each relation to the drug actually
   named in that statement.
6. If no relation matches the allowed labels, return an empty list.

Return JSON only, in this exact shape:
{{"relations": [{{"source": "", "source_type": "", "relation": "",
                  "target": "", "target_type": ""}}]}}

ABSTRACT:
{texte}
"""
print("Prompt défini.")

Prompt défini.


In [9]:
import os
print([f for f in os.listdir("/content") if f.endswith(".json")])

['controle_imatinib.json', 'dry_run_sample.json']


In [10]:
import json, time, requests

def extraire(texte):
    debut = time.time()
    reponse = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json={
            "model": "qwen2.5:7b",
            "prompt": PROMPT.format(texte=texte),
            "format": "json",
            "stream": False,
            "options": {"temperature": 0, "num_ctx": 4096, "num_predict": 800},
        },
        timeout=600,
    )
    return reponse.json()["response"], time.time() - debut

with open("controle_imatinib.json", encoding="utf-8") as fichier:
    controle = json.load(fichier)

# On ne teste que les articles portant le saut 2, seul en cause
saut2 = [a for a in controle
         if any(p["role"] == "cible" and p["saut"] == 2 for p in a["provenance"])]

print(f"Articles portant le saut 2 : {len(saut2)}\n")
for article in saut2:
    texte = article["titre"] + "\n\n" + article["resume"]
    brut, duree = extraire(texte)
    try:
        relations = json.loads(brut).get("relations", [])
    except Exception:
        relations = []
    print("=" * 72)
    print(f"PMID {article['pmid']} ({article['annee']}) — {duree:.1f}s")
    print(f"{article['titre'][:68]}")
    print("  ATTENDU : BCR/ABL --causes--> CML (ph+)")
    print("  EXTRAIT :")
    if not relations:
        print("    (aucune)")
    for rel in relations:
        print(f"    [{rel.get('source_type')}] {str(rel.get('source'))[:26]} "
              f"--{rel.get('relation')}--> "
              f"[{rel.get('target_type')}] {str(rel.get('target'))[:26]}")

Articles portant le saut 2 : 4

PMID 19108785 (2008) — 96.9s
Nilotinib: a second-generation tyrosine kinase inhibitor for the tre
  ATTENDU : BCR/ABL --causes--> CML (ph+)
  EXTRAIT :
    [Drug] nilotinib --competitively inhibits--> [Protein] cytochrome P450 (CYP) 3A4
    [Drug] nilotinib --competitively inhibits--> [Protein] cytochrome P450 (CYP) 2C8
    [Drug] nilotinib --competitively inhibits--> [Protein] cytochrome P450 (CYP) 2C9
    [Drug] nilotinib --competitively inhibits--> [Protein] cytochrome P450 (CYP) 2D6
PMID 17284091 (2007) — 2.1s
Imatinib: a review of its use in chronic myeloid leukaemia.
  ATTENDU : BCR/ABL --causes--> CML (ph+)
  EXTRAIT :
    [Drug] Imatinib --inhibits--> [Protein] BCR-Abelson (ABL) fusion p
PMID 25678499 (2015) — 10.1s
Morgana acts as an oncosuppressor in chronic myeloid leukemia.
  ATTENDU : BCR/ABL --causes--> CML (ph+)
  EXTRAIT :
    [Protein] morgana --negatively regulates--> [BiologicalProcess] centrosome duplication
    [Protein] morgana --ne

In [11]:
import subprocess, time, requests, os

def ollama_repond():
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not os.path.exists("/usr/local/bin/ollama"):
    print("Installation des prérequis et d'Ollama (~2 min)...")
    subprocess.run("apt-get install -y -qq zstd pciutils lshw", shell=True)
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)

if not ollama_repond():
    print("Démarrage du serveur...")
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                     start_new_session=True)
    for _ in range(30):
        if ollama_repond():
            break
        time.sleep(1)

print("Ollama :", "actif" if ollama_repond() else "ARRÊTÉ")

Ollama : actif


In [12]:
!ollama pull qwen2.5:7b
!ollama list


NAME          ID              SIZE      MODIFIED               
qwen2.5:7b    845dbda0ea48    4.7 GB    Less than a second ago    


In [13]:
from google.colab import drive
drive.mount('/content/drive')

import os
DOSSIER = "/content/drive/MyDrive/stage-graphrag"
os.makedirs(DOSSIER, exist_ok=True)
print("Dossier prêt :", DOSSIER)
print("Contenu :", os.listdir(DOSSIER))

Mounted at /content/drive
Dossier prêt : /content/drive/MyDrive/stage-graphrag
Contenu : []


In [14]:
import shutil, os
if os.path.exists("/content/corpus.json"):
    shutil.copy("/content/corpus.json", f"{DOSSIER}/corpus.json")
    print("Copié sur le Drive.")
print("Sur le Drive :", os.listdir(DOSSIER))

Copié sur le Drive.
Sur le Drive : ['corpus.json']


In [15]:
PROMPT = """You are a biomedical relation extraction system.

Extract relations from the abstract below.

ENTITY TYPES — use exactly these labels:
- Drug: a therapeutic agent administered to a patient
- Protein: a protein, receptor, enzyme or transporter
- ChemicalSubstance: any other molecule (endogenous compound, metabolite, hormone)
- BiologicalProcess: a biological process, pathway or molecular activity
- PhenotypicFeature: an observable symptom or clinical sign
- Disease: a named pathology or diagnosis
- OrganismTaxon: an organism (bacterium, virus, parasite)

RELATION TYPES — you MUST use one of these exact labels. Never use a verb
taken from the text as a label.

VERB TRANSLATION TABLE — find the verb in the text, then output the label:
  inhibits, blocks, antagonizes, suppresses, is an antagonist of,
  reduces the activity of              -> decreases activity of
  activates, stimulates, agonizes, is an agonist of,
  enhances the activity of             -> increases activity of
  increases the level / production / release of, upregulates the amount of
                                       -> increases abundance of
  decreases the level / production / release of, depletes
                                       -> decreases abundance of
  promotes, drives, upregulates (a process)
                                       -> positively regulates
  downregulates, impairs, dampens (a process)
                                       -> negatively regulates
  is associated with higher levels or greater severity
                                       -> positively correlated with
  is associated with lower levels, protects against
                                       -> negatively correlated with
  is involved in, takes part in, is a component of
                                       -> participates in
  is a symptom of, is a sign of, manifests as
                                       -> manifestation of
  leads to, results in, is responsible for
                                       -> causes

Use "decreases activity of" / "increases activity of" when the target is a
Protein. Use "negatively regulates" / "positively regulates" when the target
is a BiologicalProcess.

RULES:
1. Extract ONLY relations explicitly stated in the abstract. Never infer.
2. Write entity names exactly as they appear in the text. Do not expand abbreviations.
3. Preserve direction: the source performs the relation on the target.
4. Never invert polarity. An antagonist decreases; an agonist increases.
5. If several drugs are discussed, attribute each relation to the drug actually
   named in that statement.
6. If no relation matches the allowed labels, return an empty list.

Return JSON only, in this exact shape:
{{"relations": [{{"source": "", "source_type": "", "relation": "",
                  "target": "", "target_type": ""}}]}}

ABSTRACT:
{texte}
"""
print("Prompt défini.")

Prompt défini.


In [16]:
import json, time, os, requests

CORPUS = f"{DOSSIER}/corpus.json"
SORTIE = f"{DOSSIER}/extraction.jsonl"

def extraire(texte, essais=3):
    for tentative in range(essais):
        try:
            reponse = requests.post(
                "http://127.0.0.1:11434/api/generate",
                json={
                    "model": "qwen2.5:7b",
                    "prompt": PROMPT.format(texte=texte),
                    "format": "json",
                    "stream": False,
                    "options": {"temperature": 0, "num_ctx": 4096,
                                "num_predict": 800},
                },
                timeout=600,
            )
            return reponse.json()["response"]
        except Exception as erreur:
            if tentative == essais - 1:
                raise
            print(f"    échec réseau, nouvelle tentative dans 5 s ({erreur})")
            time.sleep(5)

# --- Reprise
faits = set()
if os.path.exists(SORTIE):
    with open(SORTIE, encoding="utf-8") as fichier:
        for ligne in fichier:
            try:
                faits.add(json.loads(ligne)["pmid"])
            except Exception:
                pass

with open(CORPUS, encoding="utf-8") as fichier:
    corpus = json.load(fichier)

restants = [a for a in corpus if a["pmid"] not in faits]
print(f"Corpus : {len(corpus)} — déjà traités : {len(faits)} — "
      f"restants : {len(restants)}")

if restants:
    extraire("Warm up.")          # chauffe, hors chronomètre
    print("Modèle chargé.\n")

debut = time.time()
for i, article in enumerate(restants, 1):
    texte = article["titre"] + "\n\n" + article["resume"]
    t0 = time.time()
    try:
        brut = extraire(texte)
    except Exception as erreur:
        print(f"\nARRÊT sur le PMID {article['pmid']} : {erreur}")
        print("Les résultats déjà obtenus sont conservés. "
              "Relancer la cellule 1 puis celle-ci.")
        break
    duree = time.time() - t0
    try:
        relations = json.loads(brut).get("relations", [])
        erreur_json = None
    except Exception as e:
        relations, erreur_json = [], str(e)

    with open(SORTIE, "a", encoding="utf-8") as fichier:
        fichier.write(json.dumps({
            "pmid": article["pmid"],
            "duree": round(duree, 2),
            "relations": relations,
            "erreur": erreur_json,
            "brut": brut,
        }, ensure_ascii=False) + "\n")

    if i % 25 == 0 or i == len(restants):
        ecoule = time.time() - debut
        reste = (ecoule / i) * (len(restants) - i)
        print(f"  {i}/{len(restants)}  —  {ecoule/60:.1f} min écoulées,  "
              f"~{reste/60:.1f} min restantes")

print(f"\nTerminé. Fichier : {SORTIE}")

Corpus : 475 — déjà traités : 0 — restants : 475
Modèle chargé.

  25/475  —  2.3 min écoulées,  ~40.9 min restantes
  50/475  —  4.5 min écoulées,  ~38.1 min restantes
  75/475  —  7.3 min écoulées,  ~39.2 min restantes
  100/475  —  9.0 min écoulées,  ~33.7 min restantes
  125/475  —  11.5 min écoulées,  ~32.2 min restantes
  150/475  —  14.1 min écoulées,  ~30.5 min restantes
  175/475  —  16.5 min écoulées,  ~28.4 min restantes
  200/475  —  18.9 min écoulées,  ~26.0 min restantes
  225/475  —  19.9 min écoulées,  ~22.1 min restantes
  250/475  —  21.9 min écoulées,  ~19.7 min restantes
  275/475  —  23.2 min écoulées,  ~16.9 min restantes
  300/475  —  24.7 min écoulées,  ~14.4 min restantes
  325/475  —  25.9 min écoulées,  ~12.0 min restantes
  350/475  —  27.1 min écoulées,  ~9.7 min restantes
  375/475  —  28.7 min écoulées,  ~7.7 min restantes
  400/475  —  29.7 min écoulées,  ~5.6 min restantes
  425/475  —  31.4 min écoulées,  ~3.7 min restantes
  450/475  —  33.7 min écoul